In [83]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.cluster import AgglomerativeClustering
from sklearn.manifold import TSNE
from scipy.cluster.hierarchy import dendrogram, linkage
import os

In [84]:
# Duomenų rinkiniai ir klasterių skaičiai pagal:
# Empirinis, Alkūnės, Vidutinio silueto metodus
cluster_counts = {
    "norm_full.csv": [76, 6, 3, 9],
    "norm_6.csv": [27, 7, 13, 8],
    "dvi_dim.csv": [27, 10, 11, 10]
}
cluster_labels = ["empirinis", "alkunes", "silueto", "dendograma"]

base_folder = "hierarchical"  # Pagrindinis aplankas rezultatams
linkage_method = "ward"       # Naudojamas sujungimo metodas
metric_type = "euclidean"     # Naudojama metrika


In [85]:
for file in cluster_counts.keys():
    print(f"\n🔹 Apdorojamas duomenų failas: {file}")
    df = pd.read_csv(file)

    if 'label' in df.columns:
        X = df.drop(columns=['label']).values
    else:
        X = df.values

    print(f"Duomenys nuskaityti: {file}")

    # Sukuriamas atitinkamas aplankas
    file_folder = os.path.join(base_folder, file[:-4])
    os.makedirs(file_folder, exist_ok=True)

    # t-SNE nustatymai
    if file == "dvi_dim.csv":
        X_tsne = np.array(X)
    elif file == "norm_full.csv":
        perp, learn_r, early_ex = 50, 200, 24
    elif file == "norm_6.csv":
        perp, learn_r, early_ex = 50, 50, 12

    if file != "dvi_dim.csv":
        tsne = TSNE(
            n_components=2,         # sumažiname duomenų dimensijų skaičių iki 2
            perplexity=perp,        # kiek artimiausių kaimynų laikoma reikšmingais
            learning_rate=learn_r,  # mokymosi greitis
            max_iter=1000,          # iteracijų skaičius
            early_exaggeration=early_ex,  # padeda atskirti klasterius
            metric="euclidean",     # atstumo metrika
            random_state=67,        # atkuriamumas
            init="pca"              # pradinė taškų pozicija nustatoma PCA pagrindu
        )
        X_tsne = tsne.fit_transform(X)

    # Dendograma
    print("Kuriama dendrograma...")
    linked = linkage(X, method=linkage_method, metric=metric_type)

    plt.figure(figsize=(12, 6))
    dendrogram(
        linked,
        orientation='top',
        distance_sort='descending',
        show_leaf_counts=False
    )
    plt.title(f"{file} — Hierarchinio klasterizavimo dendrograma\n({linkage_method.capitalize()} + {metric_type.capitalize()})")
    plt.xlabel("Duomenų taškai")
    plt.ylabel("Atstumas")
    plt.tight_layout()
    plt.savefig(os.path.join(file_folder, f"dendrogram_{file[:-4]}.png"), dpi=200)
    plt.close()

    # t-SNE vizualizacijos kiekvienam klasterių skaičiui
    print("Kuriami t-SNE grafikai...")
    auto_cluster_counts = cluster_counts[file]
    auto_cluster_labels = cluster_labels

    for label, k in zip(auto_cluster_labels, auto_cluster_counts):
        print(f"{label} metodas (k={k})")

        agg = AgglomerativeClustering(
            n_clusters=k,
            linkage=linkage_method,
            metric=metric_type
        )
        y_pred = agg.fit_predict(X)

        # Vizualizacija
        plt.figure(figsize=(9, 9))
        cmap = ListedColormap(
            list(plt.cm.tab20.colors) +
            list(plt.cm.tab20b.colors) +
            list(plt.cm.tab20c.colors)
        )
        plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_pred, cmap=cmap, s=50)
        plt.title(f"{file} — t-SNE pagal hierarchinius klasterius\n({label}, k={k})")
        plt.xlabel("t-SNE komponentė 1")
        plt.ylabel("t-SNE komponentė 2")
        plt.grid(True)
        plt.tight_layout()

        tsne_path = os.path.join(file_folder, f"tsne_{file[:-4]}_{label}_{k}.png")
        plt.savefig(tsne_path, dpi=200)
        plt.close()

    print(f"Grafikai išsaugoti aplanke: {file_folder}")

print("\n Duomenų rinkiniai apdoroti ir išsaugoti.")


🔹 Apdorojamas duomenų failas: norm_full.csv
Duomenys nuskaityti: norm_full.csv
Kuriama dendrograma...
Kuriami t-SNE grafikai...
empirinis metodas (k=76)
alkunes metodas (k=6)
silueto metodas (k=3)
dendograma metodas (k=9)
Grafikai išsaugoti aplanke: hierarchical/norm_full

🔹 Apdorojamas duomenų failas: norm_6.csv
Duomenys nuskaityti: norm_6.csv
Kuriama dendrograma...
Kuriami t-SNE grafikai...
empirinis metodas (k=27)
alkunes metodas (k=7)
silueto metodas (k=13)
dendograma metodas (k=8)
Grafikai išsaugoti aplanke: hierarchical/norm_6

🔹 Apdorojamas duomenų failas: dvi_dim.csv
Duomenys nuskaityti: dvi_dim.csv
Kuriama dendrograma...
Kuriami t-SNE grafikai...
empirinis metodas (k=27)
alkunes metodas (k=10)
silueto metodas (k=11)
dendograma metodas (k=10)
Grafikai išsaugoti aplanke: hierarchical/dvi_dim

 Duomenų rinkiniai apdoroti ir išsaugoti.


In [ ]:
from sklearn.metrics import silhouette_score

score_orig = silhouette_score(X, y_kmeans)
score_tsne = silhouette_score(X_tsne, y_kmeans)

print("Silueto koef. originalioje erdvėje:", score_orig)
print("Silueto koef. po t-SNE:", score_tsne)